# Summary Universe

Provider-agnostic ticker universe table. One row per instrument with `stooq_ticker`
and `yahoo_ticker` columns for per-provider symbol translation.

Two CLI steps:
- `seed-universe` — reads `data/stooq/raw/markets.csv`, writes editable `data/universe.csv`
- `universe` — reads `data/universe.csv`, writes the DB table

Translation rules applied at seed time:
- Currencies 6-char alpha (`EURUSD`) → `EURUSD=X`
- Currencies non-standard (`NOK_I`, `EUR_I`) → NULL (Stooq-specific, no Yahoo equivalent)
- Stooq stocks indices (`^_UK`, `^_US`) → NULL (Stooq-proprietary basket indices)
- Preferred shares `BASE_X` → `BASE-PX` (e.g. `RNR_F` → `RNR-PF`)
- All others → same as `Ticker`

In [1]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.universe import universe

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Table

Schema, row count, and key stats for the `universe` table.
Rebuilt on demand — does not update automatically when sources are stored.

### `universe`

One row per instrument (canonical key = Stooq ticker symbol, without exchange suffix).

| Column | Type | Notes |
|---|---|---|
| Ticker | VARCHAR | Canonical key — Stooq SrcId stripped of suffix (e.g. AAPL) |
| Market | VARCHAR | Market category from Stooq zip structure |
| stooq_ticker | VARCHAR | Full Stooq source ticker / SrcId (e.g. AAPL.US); NULL if not Stooq-sourced |
| yahoo_ticker | VARCHAR | yfinance symbol; NULL when no Yahoo equivalent |

In [2]:
_sample = universe('AAPL')
display(_sample.dtypes.to_frame('dtype'))
display(_sample.T)

InvalidInputException: Invalid Input Error: Python Object "universe" of type "function" found on line "/mnt/Dev/active_python_projects/investment_research_platform/src/irp/data/universe.py:51" not suitable for replacement scans.
Make sure that "universe" is either a pandas.DataFrame, duckdb.DuckDBPyRelation, pyarrow Table, Dataset, RecordBatchReader, Scanner, or NumPy ndarrays with supported format

In [ ]:
_stats = db().execute("""
    SELECT
        COUNT(*)                         AS tickers,
        COUNT(DISTINCT Market)           AS markets,
        SUM(stooq_ticker IS NOT NULL)    AS have_stooq_ticker,
        SUM(yahoo_ticker IS NOT NULL)    AS have_yahoo_ticker,
        SUM(yahoo_ticker IS NULL)        AS null_yahoo_ticker
    FROM universe
""").df().T
_stats.columns = ['universe']
display(_stats)

## Coverage by Market

How many tickers per market have a valid `yahoo_ticker` vs NULL.

In [ ]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*)                                                    AS tickers,
        SUM(yahoo_ticker IS NOT NULL)                               AS have_yahoo,
        SUM(yahoo_ticker IS NULL)                                   AS no_yahoo,
        ROUND(SUM(yahoo_ticker IS NOT NULL) * 100.0 / COUNT(*), 1) AS pct_yahoo
    FROM universe
    GROUP BY Market
    ORDER BY tickers DESC
""").df())

## Translation Breakdown

How many tickers fall into each translation category.

In [ ]:
display(db().execute("""
    SELECT
        CASE
            WHEN yahoo_ticker IS NULL              THEN 'null (no Yahoo equivalent)'
            WHEN yahoo_ticker LIKE '%=X'           THEN 'currency (=X suffix)'
            WHEN yahoo_ticker LIKE '%-P_'          THEN 'preferred share (-PX)'
            WHEN yahoo_ticker = Ticker             THEN 'unchanged'
            ELSE                                        'other'
        END                                            AS translation,
        COUNT(*)                                       AS tickers
    FROM universe
    GROUP BY 1
    ORDER BY tickers DESC
""").df())

## NULL yahoo_ticker Detail

Which instruments have no Yahoo equivalent and why.

In [ ]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*) AS null_tickers,
        STRING_AGG(Ticker, ', ' ORDER BY Ticker) AS examples
    FROM universe
    WHERE yahoo_ticker IS NULL
    GROUP BY Market
    ORDER BY null_tickers DESC
""").df())

## Translated Tickers Sample

Sample of tickers where `yahoo_ticker` differs from `Ticker`.

In [ ]:
print('=== Currency translations (first 10) ===')
display(db().execute("""
    SELECT Ticker, Market, yahoo_ticker
    FROM universe
    WHERE yahoo_ticker LIKE '%=X'
    ORDER BY Ticker
    LIMIT 10
""").df())

print('=== Preferred share translations (first 10) ===')
display(db().execute("""
    SELECT Ticker, Market, yahoo_ticker
    FROM universe
    WHERE yahoo_ticker LIKE '%-P_'
    ORDER BY Ticker
    LIMIT 10
""").df())

## Market Distribution

Ticker count per market category.

In [ ]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*)                                    AS tickers,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM universe
    GROUP BY Market
    ORDER BY tickers DESC
""").df())